In [7]:
# ==============================================================================
# GLOBAL CONFIGURATION & PATHS 
# ==============================================================================
INPUT_PATH  = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
OUTPUT_PATH = "/kaggle/working"

import os
import gc
import torch 
import numpy as np
import polars as pl
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

# ==============================================================================
# 1. POLARS HIGH-PERFORMANCE FEATURE ENGINEERING (PATCHED FOR TEST SET)
# ==============================================================================
def extract_polars_features(df_path: str, typewell_path: str, is_train: bool):
    """
    Replaces slow Pandas rolling windows with lightning-fast Polars expressions.
    """
    q = pl.scan_csv(df_path, null_values=["", "NA", "NaN"])
    
    # --- FIXED: DYNAMIC COLUMN CASTING ---
    # Only try to cast 'TVT' if we are building the training set
    numeric_cols = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
    if is_train:
        numeric_cols.append("TVT")
        
    q = q.with_columns([
        pl.col(c).cast(pl.Float32, strict=False) for c in numeric_cols
    ])
    # -------------------------------------
    
    # Calculate Geometric Derivatives and Rolling GR Stats natively in Rust
    q = q.with_columns([
        (pl.col("Z").diff() / pl.col("MD").diff().fill_null(1.0)).alias("dz_dmd"),
        pl.col("GR").rolling_mean(window_size=5, center=True).alias("gr_roll5"),
        pl.col("GR").rolling_std(window_size=5, center=True).alias("gr_std5"),
        pl.col("GR").diff().alias("gr_grad1"),
        pl.col("GR").diff().diff().alias("gr_grad2"),
        pl.col("GR").shift(-5).alias("gr_lead5"), 
        pl.col("GR").shift(-10).alias("gr_lead10")
    ])
    
    df = q.collect()
    
    mask_start = df.with_columns(pl.col("TVT_input").is_null().alias("is_hidden")) \
                   .select(pl.arg_where(pl.col("is_hidden")).first()).item()
    
    if mask_start is None or mask_start == 0:
        return None # Invalid well
        
    known = df.slice(0, mask_start)
    hidden = df.slice(mask_start, len(df))
    
    if is_train:
        hidden = hidden.filter(pl.col("TVT").is_not_null())
        if len(hidden) == 0: return None
        
    lkt = known["TVT_input"][-1]
    last_known_z = known["Z"][-1]
    
    hidden = hidden.with_columns([
        (pl.col("Z") - last_known_z).alias("dz_from_last"),
        (pl.col("MD") - known["MD"][-1]).alias("dmd_from_last")
    ])
    
    tw = pl.read_csv(typewell_path, null_values=["", "NA", "NaN"])
    tw = tw.with_columns([
        pl.col("GR").cast(pl.Float32, strict=False),
        pl.col("TVT").cast(pl.Float32, strict=False)
    ])
    
    tw_gr_mean = tw["GR"].mean()
    tw_tvt_range = tw["TVT"].max() - tw["TVT"].min()
    
    well_name = Path(df_path).name.split("__")[0]
    
    cols_to_select = [
        pl.lit(well_name).alias("well"),
        (pl.lit(well_name) + "_" + pl.col("MD").cast(pl.Utf8)).alias("prediction_id"),
        pl.col("MD"), pl.col("X"), pl.col("Y"), pl.col("Z"),
        pl.col("dz_dmd"), pl.col("gr_roll5"), pl.col("gr_std5"), 
        pl.col("gr_grad1"), pl.col("gr_grad2"), pl.col("gr_lead5"), pl.col("gr_lead10"),
        pl.col("dz_from_last"), pl.col("dmd_from_last"),
        pl.lit(lkt).alias("last_known_tvt"),
        pl.lit(tw_gr_mean).alias("tw_gr_global_mean"),
        pl.lit(tw_tvt_range).alias("tw_tvt_range")
    ]
    
    if is_train:
        cols_to_select.append((pl.col("TVT") - lkt).alias("target"))
        
    result = hidden.select(cols_to_select)
        
    return result

# ==============================================================================
# 2. DATASET COMPILATION WITH MEMORY SAFEGUARDS
# ==============================================================================
def build_memory_safe_dataset(data_dir: Path, is_train: bool):
    wells = list(data_dir.glob("*__horizontal_well.csv"))
    df_list = []
    
    print(f"Processing {len(wells)} wells from {data_dir.name} using Polars...")
    for hw_path in wells:
        tw_path = str(hw_path).replace("__horizontal_well", "__typewell")
        # Safety check in case a typewell is missing
        if not Path(tw_path).exists():
            continue
            
        feat_df = extract_polars_features(str(hw_path), tw_path, is_train)
        if feat_df is not None:
            df_list.append(feat_df)
            
    final_df = pl.concat(df_list).to_pandas()
    
    # Create Group IDs for GroupKFold
    unique_wells = final_df['well'].unique()
    well_map = {w: i for i, w in enumerate(unique_wells)}
    final_df['group_id'] = final_df['well'].map(well_map)
    
    # Downcast 64-bit floats to 32-bit to save ~50% RAM/VRAM
    float_cols = final_df.select_dtypes(include=['float64']).columns
    final_df[float_cols] = final_df[float_cols].astype('float32')
    
    return final_df

# ==============================================================================
# 3. GPU-ACCELERATED MODELING & INFERENCE
# ==============================================================================
def train_and_infer_t4():
    train_dir = Path(INPUT_PATH) / "train"
    test_dir  = Path(INPUT_PATH) / "test"
    
    train_df = build_memory_safe_dataset(train_dir, is_train=True)
    test_df  = build_memory_safe_dataset(test_dir, is_train=False)
    
    features = [c for c in train_df.columns if c not in ['well', 'prediction_id', 'target', 'group_id']]
    
    X = train_df[features]
    y = train_df['target']
    groups = train_df['group_id']
    
    # XGBoost Config optimized for T4
    xgb_params = {
        'n_estimators': 2500,
        'learning_rate': 0.03,
        'max_depth': 7,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'tree_method': 'hist',
        'device': 'cuda', 
        'eval_metric': 'rmse',
        'random_state': 42
    }
    
    cv = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(train_df), dtype=np.float32)
    test_preds = np.zeros(len(test_df), dtype=np.float32)
    
    print("\nInitiating GPU T4 Training Sequence with Memory Safeguards...")
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups=groups)):
        print(f"--- Training Fold {fold + 1} ---")
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        
        model = xgb.XGBRegressor(**xgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=250)
        
        oof_preds[va_idx] = model.predict(X_va)
        test_preds += model.predict(test_df[features]) / cv.n_splits
        
        # Save checkpoint to prevent timeout loss
        model.save_model(f"{OUTPUT_PATH}/xgb_fold_{fold}.json")
        
        # EXTREME GARBAGE COLLECTION
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    print(f"\nFinal Out-Of-Fold RMSE: {root_mean_squared_error(y, oof_preds):.5f}")
    
    # Reconstruct absolute TVT
    test_df['tvt'] = test_preds + test_df['last_known_tvt']
    
    # --- CONTINGENCY PLAN: THE SAFE FALLBACK ---
    nan_count = test_df['tvt'].isna().sum()
    if nan_count > 0:
        print(f"WARNING: Caught {nan_count} NaN predictions. Applying Safe Fallback.")
        test_df['tvt'] = test_df['tvt'].fillna(test_df['last_known_tvt'])
    
    # Save Output
    sub = test_df[['prediction_id', 'tvt']].rename(columns={'prediction_id': 'id'})
    sub.to_csv(os.path.join(OUTPUT_PATH, "submission.csv"), index=False)
    print(f"Submission saved to {OUTPUT_PATH}/submission.csv")

# Execute the pipeline
if __name__ == "__main__":
    train_and_infer_t4()

Processing 773 wells from train using Polars...
Processing 3 wells from test using Polars...

Initiating GPU T4 Training Sequence with Memory Safeguards...
--- Training Fold 1 ---
[0]	validation_0-rmse:15.31205
[250]	validation_0-rmse:15.31977
[500]	validation_0-rmse:15.37981
[750]	validation_0-rmse:15.31323
[1000]	validation_0-rmse:15.27673
[1250]	validation_0-rmse:15.23477
[1500]	validation_0-rmse:15.20099
[1750]	validation_0-rmse:15.18184
[2000]	validation_0-rmse:15.16338
[2250]	validation_0-rmse:15.14744
[2499]	validation_0-rmse:15.13378


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [04:20:44] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


--- Training Fold 2 ---
[0]	validation_0-rmse:15.15768
[250]	validation_0-rmse:14.65827
[500]	validation_0-rmse:14.67593
[750]	validation_0-rmse:14.63147
[1000]	validation_0-rmse:14.56777
[1250]	validation_0-rmse:14.52402
[1500]	validation_0-rmse:14.51356
[1750]	validation_0-rmse:14.51766
[2000]	validation_0-rmse:14.50679
[2250]	validation_0-rmse:14.49867
[2499]	validation_0-rmse:14.48962
--- Training Fold 3 ---
[0]	validation_0-rmse:14.94876
[250]	validation_0-rmse:14.75610
[500]	validation_0-rmse:14.77009
[750]	validation_0-rmse:14.68538
[1000]	validation_0-rmse:14.62441
[1250]	validation_0-rmse:14.58712
[1500]	validation_0-rmse:14.57050
[1750]	validation_0-rmse:14.54816
[2000]	validation_0-rmse:14.53492
[2250]	validation_0-rmse:14.52778
[2499]	validation_0-rmse:14.52072
--- Training Fold 4 ---
[0]	validation_0-rmse:16.32884
[250]	validation_0-rmse:15.94628
[500]	validation_0-rmse:15.82713
[750]	validation_0-rmse:15.73203
[1000]	validation_0-rmse:15.65884
[1250]	validation_0-rmse:15.